# politics - us dataset
fine-tuning launch/POLITICS on stanford congressional speech
POLITICS is roberta-based — using auto classes

In [ ]:
# install packages
! pip install transformers scikit-learn 

In [ ]:
# using auto classes since politics is roberta-based
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# using auto classes as politics is roberta-based
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
device =torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# file paths
train_path= '/kaggle/input/datasets/zainabrizvi5/thesis-data/final_dataset/us_train.csv'
val_path = '/kaggle/input/datasets/zainabrizvi5/thesis-data/final_dataset/us_val.csv'
test_path= '/kaggle/input/datasets/zainabrizvi5/thesis-data/final_dataset/us_test.csv'
output_dir = '/kaggle/working/politics_us'

In [ ]:
model_name ='launch/POLITICS'
max_length=256
batch_size=16
epochs = 3
lr = 2e-5

In [ ]:
#binary labels
label2id ={'left': 0, 'right': 1}
id2label= {0: 'left', 1: 'right'}

In [ ]:
# load data
train_df= pd.read_csv(train_path)
val_df=pd.read_csv(val_path)
test_df = pd.read_csv(test_path)
print(f'train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}')

In [ ]:
# dataset class
class SpeechDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, label2id):
        self.texts= df['text'].tolist()
        self.labels= [label2id[l] for l in df['label'].tolist()]
        self.tokenizer= tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx],max_length=self.max_length,padding='max_length',truncation=True,return_tensors='pt')
        return {'input_ids':enc['input_ids'].squeeze(),'attention_mask': enc['attention_mask'].squeeze(),'label': torch.tensor(self.labels[idx])}

In [ ]:
# auto tokenizer handles roberta tokenization automatically
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# create datasets
train_dataset = SpeechDataset(train_df,tokenizer, max_length, label2id)
val_dataset= SpeechDataset(val_df,tokenizer, max_length, label2id)
test_dataset= SpeechDataset(test_df,tokenizer, max_length, label2id)

In [ ]:
# dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader= DataLoader(val_dataset,batch_size=batch_size)
test_loader = DataLoader(test_dataset,batch_size=batch_size)

In [ ]:
# load the politics model
model = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=2, id2label=id2label,label2id=label2id)
model.to(device)
print('model loaded')

In [ ]:
#optimizer and scheduler
optimizer= AdamW(model.parameters(), lr=lr)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [ ]:
#evaluation
def evaluate(model,loader, split='val'):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask= batch['attention_mask'].to(device)
            labs = batch['label'].to(device)
            out =model(input_ids=ids, attention_mask=mask)
            preds =torch.argmax(out.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labs.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1= f1_score(all_labels, all_preds, average='macro')
    
    print(f'{split} — acc: {acc:.4f}, macro f1: {f1:.4f}')
    return acc, f1

In [ ]:
# training loop
history =[]
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        ids= batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['label'].to(device)
        out= model(input_ids=ids, attention_mask=mask, labels=labs)
        loss = out.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    avg_loss = total_loss/len(train_loader)
    val_acc, val_f1 = evaluate(model, val_loader, split=f'epoch {epoch+1} val')
    history.append({'epoch': epoch+1,'train_loss': avg_loss,'val_acc': val_acc, 'val_f1': val_f1})
    print(f'epoch {epoch+1} train loss: {avg_loss:.4f}')

In [ ]:
# history
print('training done')
for h in history:
    print(h)

In [ ]:
# test evaluation
print('final test evaluation')
test_acc, test_f1 = evaluate(model, test_loader, split='test')

In [ ]:
# save model and results
pd.DataFrame(history).to_csv(os.path.join(output_dir, 'training_history.csv'), index=False)
pd.DataFrame([{'model': 'POLITICS', 'dataset': 'US (Stanford)', 'test_acc': test_acc, 'test_f1': test_f1,'epochs': epochs, 'batch_size': batch_size, 'lr': lr, 'max_length': max_length}]).to_csv(os.path.join(output_dir, 'final_results.csv'), index=False)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'saved to {output_dir}')